## PHASE 4 : Software Architecture & Deployment

Following the reconstruction analysis and clinical validation completed in Phase 3, this phase organises the workflow into a modular software structure. The reconstruction logic is encapsulated into a clear interface that can be executed programmatically or integrated into external applications.

Phase 4 introduces an object‑oriented reconstruction engine, a FastAPI endpoint for running the pipeline remotely, and a consistent execution environment for local or containerised use. Basic tests are included to verify that the core functionality behaves as expected.

These components complete the transition from the exploratory workflow in Phases 1–3 to a structured implementation suitable for further development and integration.

**4.1 Importing and Using the Reconstruction Engine**

This section introduces the reconstruction engine used throughout Phase 4. The forward operator, adjoint operator, and FISTA–TV solver are implemented within a single class to provide a clear interface for running the reconstruction pipeline.

The example below illustrates how the engine is imported and instantiated. In practice, this class forms the core computational component used by the FastAPI service and the accompanying tests defined later in this phase.

In [2]:
# Import the reconstruction engine from the standalone module
from compressed_sensing_mri import CompressedSensingMRI
import numpy as np

# Example usage (placeholder demonstration)

# Create a dummy mask for demonstration
mask = np.ones((256, 256), dtype=np.float32)

# Instantiate the reconstruction engine
engine = CompressedSensingMRI(mask=mask)

# Create dummy k-space data (normally provided by the API or dataset)
kspace = np.zeros((256, 256), dtype=np.complex64)

# Run reconstruction
recon = engine.reconstruct(kspace)

# Display shape to confirm successful execution
recon.shape


(256, 256)

**4.2 FastAPI Inference Service: `/reconstruct` Endpoint**

This section defines a FastAPI endpoint for running the reconstruction pipeline on externally provided k‑space data. The `/reconstruct` route receives the real and imaginary components of k‑space, a sampling mask, and the original image shape. These inputs are converted into NumPy arrays and passed to the reconstruction engine introduced in Section 4.1.

Pydantic models are used to structure the request and response formats, and basic validation is included to ensure that the input dimensions are consistent. The implementation shown here corresponds to the standalone `api_service.py` module, which can be run independently when deploying the service.

In [3]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import numpy as np
import time


# Import reconstruction engine from standalone module
from compressed_sensing_mri import CompressedSensingMRI

# Pydantic request model
class ReconstructionRequest(BaseModel):
    kspace_real: list      # flattened real part of k-space
    kspace_imag: list      # flattened imaginary part of k-space
    mask: list             # flattened sampling mask (0/1)
    shape: tuple           # original 2D shape (H, W)


# Pydantic response model
class ReconstructionResponse(BaseModel):
    image: list            # reconstructed magnitude image (2D list)
    duration_ms: float     # reconstruction time in milliseconds


# Initialise FastAPI application
app = FastAPI()


# /reconstruct endpoint
@app.post("/reconstruct", response_model=ReconstructionResponse)
def reconstruct_endpoint(payload: ReconstructionRequest):


    # Extract shape and validate dimensions
    try:
        H, W = payload.shape
        expected_len = H * W

        if not (
            len(payload.kspace_real) == expected_len and
            len(payload.kspace_imag) == expected_len and
            len(payload.mask) == expected_len
        ):
            raise ValueError("Input arrays do not match the specified shape.")

    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Invalid shape: {str(e)}")


    # Convert lists to numpy arrays and reshape
    try:
        k_real = np.array(payload.kspace_real, dtype=np.float32).reshape(H, W)
        k_imag = np.array(payload.kspace_imag, dtype=np.float32).reshape(H, W)
        kspace = k_real + 1j * k_imag

        mask = np.array(payload.mask, dtype=np.float32).reshape(H, W)

    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Array conversion failed: {str(e)}")


    # Instantiate reconstruction engine
    engine = CompressedSensingMRI(mask=mask)


    # Perform reconstruction with timing
    try:
        start = time.time()
        recon = engine.reconstruct(kspace)
        duration_ms = (time.time() - start) * 1000

    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Reconstruction failed: {str(e)}")

    # Convert output to Python list for JSON response
    recon_list = recon.tolist()

    return ReconstructionResponse(
        image=recon_list,
        duration_ms=duration_ms
    )


**4.3 Docker Container for Reproducible Deployment**

This section provides the Dockerfile used to run the reconstruction service in a containerised environment. The file installs the required dependencies, copies the reconstruction engine and API module, and starts the FastAPI application using Uvicorn. This allows the service to be executed in a consistent environment without relying on the notebook.

In [ ]:
# Base image: Python runtime
FROM python:3.10-slim

# Set working directory inside the container
WORKDIR /app

# Copy dependency list and install packages
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code into the container
COPY compressed_sensing_mri.py .
COPY api_service.py .

# Expose FastAPI port
EXPOSE 8000

# Start the FastAPI server using uvicorn
CMD ["uvicorn", "api_service:app", "--host", "0.0.0.0", "--port", "8000"]


**4.4 Pytest Unit Test for Reconstruction Engine**

This section documents a Pytest script used to verify that the reconstruction engine behaves as expected under controlled conditions. The test checks that the module imports correctly, accepts synthetic k‑space input, returns an output of the correct shape, and produces finite, non‑zero values.

The test is implemented in `tests/test_reconstruction.py` and is executed externally using Pytest.

In [4]:
# Pytest for Compressed Sensing MRI Engine

import numpy as np
from compressed_sensing_mri import CompressedSensingMRI

def test_reconstruction():
    """
    Basic test to ensure the reconstruction engine:
    - accepts valid synthetic k-space input
    - returns an output of the correct shape
    - produces non-zero magnitude values
    """

    # Create a small synthetic test image (8x8)
    H, W = 8, 8
    synthetic_image = np.ones((H, W), dtype=np.float32)


    # Forward FFT to generate synthetic k-space
    kspace = np.fft.fft2(synthetic_image)


    # Undersampling mask (fully sampled for test)
    mask = np.ones((H, W), dtype=np.float32)

    # Instantiate reconstruction engine
    engine = CompressedSensingMRI(mask=mask)

    # Perform reconstruction
    recon = engine.reconstruct(kspace)

    # Assertions: shape, type, non-zero content
    assert recon.shape == (H, W), "Reconstruction returned incorrect shape."
    assert np.isfinite(recon).all(), "Reconstruction contains invalid values."
    assert recon.sum() > 0, "Reconstruction returned all zeros."


Phase 4 provides a structured implementation of the reconstruction workflow developed in earlier phases. The reconstruction engine, API module, Docker configuration, and test script together define the core components required to run the pipeline outside the notebook environment. These elements form a clear reference for how the reconstruction method can be executed programmatically or incorporated into other systems.